In [1]:
import os
import sys
sys.dont_write_bytecode = True

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

# api keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = os.getenv("BASE_URL")

# model selection
SMALL_MODEL_NAME = "qwen/qwen3-4b"

LARGE_MODEL_NAME_V1 = "qwen/qwen3.5-122b-a10b"
LARGE_MODEL_NAME_V2 = "qwen/qwen3.5-35b-a3b"

JUDGE_MODEL_NAME_V1 = "openai/gpt-5-mini"
JUDGE_MODEL_NAME_V2 = "anthropic/claude-sonnet-4.5"

SLM_AS_ROUTER_V1 = "qwen/qwen3-1.7b"
SLM_AS_ROUTER_V2 = "qwen/qwen3-0.6b-04-28"

# concurency level
RPS = 5

# additional configuration
DATA_LOCATION_PATH = "generated/slm_flow_df"
TOKENIZER_NAME = "o200k_base"
SPACY_NLP_MODEL = "en_core_web_lg"

# policies
SLM_AS_ROUTER_CONFIDENCE_THRESHOLD = 4

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

from core.data import RAGSyntheticDataset
from core.pipeline import RAGPipelineRunner, JScore
from core.messaging import LangchainMessageBuilder, PROMPT_REGISTRY
from core.router import (
    SLMRouterOutput,
    SLMRoutingPolicy,
    WeightedRuleBasedRoutingPolicy,
    WeightedRule
)
from core.tasks import RAGTaskPrediction
from core.utils import compute_slm_routing_metrics, dump_to_csv

In [3]:
dataset = RAGSyntheticDataset.from_files(DATA_LOCATION_PATH)

In [8]:
dataset[:3]

[DatasetRecord(task='context_compression', domain='history', difficulty='complex', usage_metadata={'input_tokens': 792, 'output_tokens': 2446, 'total_tokens': 3238, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 1792}}, sample=StandardSample(query='What strategic aims and logistical constraints drove the Ottoman decision to besiege Vienna in 1683?', documents=[RAGDocument(idx=1, content='In the spring of 1683 the Ottoman leadership, under Grand Vizier Kara Mustafa, turned its attention toward Vienna as part of a broader strategy to consolidate control over Hungary and the Danube corridor. Contemporary advisers argued that seizing Vienna would decisively alter the balance in Central Europe, neutralize Habsburg capacity to threaten Ottoman frontiers, and create leverage for a favorable negotiated peace. Ottoman dispatches emphasize seeking a decisive victory rather than protracted border skirmishing.', reasoning_trace=None), RAGDoc

In [ ]:
_base_rules = [
    WeightedRule(
        name="query_token_count",
        operator="ge",
        threshold=40,
        weight=0.10,
    ),
    WeightedRule(
        name="query_noun_chunk_count",
        operator="ge",
        threshold=5,
        weight=0.15,
    ),
    WeightedRule(
        name="query_avg_word_frequency",
        operator="le",
        threshold=3.5,
        weight=0.20,
    ),
    WeightedRule(
        name="avg_lexical_overlap",
        operator="le",
        threshold=0.15,
        weight=0.25,
    ),
]
reranking_task_rules = _base_rules + [
    WeightedRule(
        name="min_lexical_overlap",
        operator="le",
        threshold=0.05,
        weight=0.15,
    ),
    WeightedRule(
        name="documents_cosine_similarity",
        operator="ge",
        threshold=0.80,
        weight=0.25,
    ),
    WeightedRule(
        name="documents_count",
        operator="ge",
        threshold=10,
        weight=0.10,
    ),
]
compression_task_rules = _base_rules + [
    WeightedRule(
        name="total_context_token_count",
        operator="ge",
        threshold=4000,
        weight=0.30,
    ),
    WeightedRule(
        name="avg_chunk_token_count",
        operator="ge",
        threshold=500,
        weight=0.15,
    ),
    WeightedRule(
        name="relevant_documents_ratio",
        operator="le",
        threshold=0.30,
        weight=0.25,
    ),
]
message_builder = LangchainMessageBuilder.from_sequence(
    ("reranking", PROMPT_REGISTRY.reranking_inference, RAGTaskPrediction),
    ("context_compression", PROMPT_REGISTRY.context_compression_inference, RAGTaskPrediction),
    ("judge", PROMPT_REGISTRY.evaluation, JScore)
)
model_kwargs = {
    "api_key": OPENROUTER_API_KEY,
    "base_url": BASE_URL,
    "max_tokens": 4092,
    "temperature": 0.0,
    "rate_limiter": InMemoryRateLimiter(
        requests_per_second=RPS,
        check_every_n_seconds=0.1,
        max_bucket_size=RPS * 2
    )
}

In [ ]:
small_model_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    routing_mode="slm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
small_model_pipeline_result = await small_model_only_pipeline.arun(dataset)

In [ ]:
small_model_pipeline_evaluation = await small_model_only_pipeline.aevaluate(small_model_pipeline_result)

In [ ]:
large_model_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    routing_mode="llm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
large_model_pipeline_result = await large_model_only_pipeline.arun(dataset)

In [ ]:
large_model_pipeline_evaluation = await large_model_only_pipeline.aevaluate(large_model_pipeline_result)

In [ ]:
rule_based_dynamic_routing_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    routing_mode="dynamic",
    messages_builder=message_builder,
    dynamic_routing_policies={
        "reranking": WeightedRuleBasedRoutingPolicy(
            *reranking_task_rules,
            min_triggers=3,
            cumulative_weights_threshold=0.45
        ),
        "context_compression": WeightedRuleBasedRoutingPolicy(
            *compression_task_rules,
            min_triggers=3,
            cumulative_weights_threshold=0.45
        )
    },
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
rule_based_pipeline_result = await rule_based_dynamic_routing_pipeline.arun(dataset)
dump_to_csv(rule_based_pipeline_result, path="rule_based_pipeline_result")

In [ ]:
rule_based_pipeline_evaluation = await rule_based_dynamic_routing_pipeline.aevaluate(rule_based_pipeline_result)
dump_to_csv(rule_based_pipeline_evaluation, path="rule_based_pipeline_evaluation")

In [ ]:
rule_based_pipeline_routing_metrics = compute_slm_routing_metrics(rule_based_pipeline_evaluation)
dump_to_csv(rule_based_pipeline_routing_metrics, path="rule_based_pipeline_routing_metrics")

In [ ]:
slm_router_client = ChatOpenAI(model=SLM_AS_ROUTER_V1, **model_kwargs)

slm_routing_policy_messages_builder = LangchainMessageBuilder.from_sequence(
    ("slm_routing_policy", PROMPT_REGISTRY.slm_as_router, SLMRouterOutput)
)
slm_as_router_policy = SLMRoutingPolicy(
    client=slm_router_client,
    message_builder=slm_routing_policy_messages_builder,
    confidence_threshold=SLM_AS_ROUTER_CONFIDENCE_THRESHOLD
)
slm_router_based_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    routing_mode="dynamic",
    messages_builder=message_builder,
    dynamic_routing_policies={
        "reranking": slm_as_router_policy,
        "context_compression": slm_as_router_policy
    },
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
slm_router_based_pipeline_result = await slm_router_based_pipeline.arun(dataset)
dump_to_csv(slm_router_based_pipeline_result, path="slm_router_based_pipeline_result")

In [ ]:
slm_router_based_pipeline_evaluation = await slm_router_based_pipeline.aevaluate(slm_router_based_pipeline_result)
dump_to_csv(slm_router_based_pipeline_evaluation, path="slm_router_based_pipeline_evaluation")

In [ ]:
slm_router_based_pipeline_routing_metrics = compute_slm_routing_metrics(slm_router_based_pipeline_evaluation)
dump_to_csv(slm_router_based_pipeline_routing_metrics, path="slm_router_based_pipeline_routing_metrics")